In [6]:

# HEARTCARE ANALYTICS - LIGHT THEME DASHBOARD

import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
from google.colab import files
import ipywidgets as widgets

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Load dataset
filename = "heart_disease_practice_dataset.csv"

if not os.path.exists(filename):
    uploaded = files.upload()
    filename = next(iter(uploaded))

df = pd.read_csv(filename)
df = df.apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=["target"]).copy()
df["target"] = df["target"].astype(int)

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
accuracy = accuracy_score(y_test, pred) * 100

# 2. Dashboard styling
display(HTML("""
<style>
.dashboard-title {
    background: linear-gradient(110deg,#eaf2ff,#f3efff);
    border: 1px solid #dce7f8;
    padding: 26px;
    border-radius: 18px;
    color: #263d63;
    font-family: Arial,sans-serif;
}
.dashboard-title h1 { margin:0; font-size:29px; }
.dashboard-title p { color:#6b7c98; margin-bottom:0; }
.kpi-grid {
    display:grid;
    grid-template-columns:repeat(auto-fit,minmax(155px,1fr));
    gap:14px; margin:18px 0;
}
.kpi {
    background:#ffffff;
    border:1px solid #e6ebf3;
    border-radius:15px;
    padding:18px;
    box-shadow:0 3px 12px rgba(45,72,110,.05);
}
.kpi-label {color:#71809a;font-size:13px;}
.kpi-value {color:#2f5288;font-size:26px;font-weight:700;margin-top:8px;}
.section-title {
    color:#344968;
    font-size:20px;
    font-weight:700;
    margin:25px 0 8px;
}
</style>

<div class="dashboard-title">
<h1>♡ HeartCare Analytics</h1>
<p>Heart Disease ML Project · MDM Dashboard</p>
<p>Light theme · Data insights · Model evaluation</p>
</div>
"""))

# 3. KPI cards
display(HTML(f"""
<div class="kpi-grid">
  <div class="kpi">
    <div class="kpi-label">Total Records</div>
    <div class="kpi-value">{len(df):,}</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Input Features</div>
    <div class="kpi-value">{len(X.columns)}</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Test Accuracy</div>
    <div class="kpi-value">{accuracy:.1f}%</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Test Samples</div>
    <div class="kpi-value">{len(y_test)}</div>
  </div>
</div>
<div class="section-title">Dataset Insights</div>
"""))

# Shared light chart style
chart_layout = dict(
    template="plotly_white",
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    font=dict(family="Arial", color="#52627a"),
    margin=dict(l=35, r=25, t=65, b=40)
)

# 4. Target distribution
target_df = (
    df["target"].map({0:"Class 0",1:"Class 1"})
    .value_counts().rename_axis("Class")
    .reset_index(name="Records")
)

fig = px.pie(
    target_df, names="Class", values="Records",
    hole=0.62, title="Target Class Distribution",
    color_discrete_sequence=["#91B8F5","#A7DFC9"]
)
fig.update_layout(**chart_layout)
fig.show()

# 5. Age distribution
fig = px.histogram(
    df, x="age", color="target", nbins=20,
    title="Age Distribution",
    labels={"age":"Age","target":"Dataset Class"},
    color_discrete_sequence=["#91B8F5","#E7B8D8"],
    barmode="overlay", opacity=0.85
)
fig.update_layout(**chart_layout)
fig.show()

# 6. Correlation heatmap
fig = px.imshow(
    df.corr(numeric_only=True),
    text_auto=".2f",
    color_continuous_scale=[
        "#F4F7FC","#D8E5F8","#91B8F5","#657FB5"
    ],
    zmin=-1, zmax=1,
    title="Feature Correlation Heatmap",
    aspect="auto"
)
fig.update_layout(**chart_layout, height=620)
fig.show()

# 7. Confusion matrix
cm = confusion_matrix(y_test, pred, labels=[0,1])

fig = px.imshow(
    cm,
    x=["Predicted 0","Predicted 1"],
    y=["Actual 0","Actual 1"],
    text_auto=True,
    color_continuous_scale=["#F5F8FE","#B8D4FA","#648FD0"],
    title="Model Evaluation - Confusion Matrix"
)
fig.update_layout(**chart_layout)
fig.show()

# 8. Dataset table
display(HTML('<div class="section-title">Dataset Preview</div>'))
display(df.head(10).style
        .background_gradient(cmap="Blues", subset=X.columns)
        .format(precision=2))

# 9. Interactive sample prediction
display(HTML('<div class="section-title">Interactive Demo Prediction</div>'))
display(HTML(
    '<p style="color:#71809a">Adjust input features and run the demo. '
    'This output is not a medical diagnosis.</p>'
))

controls = {}

for col in X.columns:
    default = float(X[col].median())

    if col in ["sex","cp","fbs","restecg","exang",
               "slope","ca","thal"]:
        controls[col] = widgets.IntSlider(
            value=int(round(default)),
            min=int(X[col].min()),
            max=int(X[col].max()),
            description=col,
            continuous_update=False,
            style={"description_width":"100px"},
            layout=widgets.Layout(width="320px")
        )
    else:
        controls[col] = widgets.FloatSlider(
            value=default,
            min=float(X[col].min()),
            max=float(X[col].max()),
            step=0.1,
            description=col,
            continuous_update=False,
            style={"description_width":"100px"},
            layout=widgets.Layout(width="320px")
        )

button = widgets.Button(
    description="Run Demo Prediction",
    button_style="info",
    icon="check"
)
output = widgets.Output()

def run_prediction(_):
    with output:
        output.clear_output()
        sample = pd.DataFrame([{
            col: controls[col].value for col in X.columns
        }])
        result = int(model.predict(sample)[0])
        probs = model.predict_proba(sample)[0]

        display(HTML(f"""
        <div style="background:#f1f6ff;border:1px solid #dce7f8;
        border-radius:14px;padding:20px;color:#304968">
        <h3>Model output: Class {result}</h3>
        <p>Class 0 model score: {probs[0]*100:.1f}%</p>
        <p>Class 1 model score: {probs[1]*100:.1f}%</p>
        <p style="font-size:12px">Educational demonstration only;
        not a medical diagnosis.</p>
        </div>
        """))

button.on_click(run_prediction)

left = widgets.VBox([controls[c] for c in list(X.columns)[:7]])
right = widgets.VBox([controls[c] for c in list(X.columns)[7:]])

display(widgets.HBox([left, right]))
display(button, output)

print("Light theme dashboard is ready.")
print("Dataset labels are synthetic practice labels.")


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,33,0,3,128,243,1,0,191,1,0.60,0,0,2,0
1,66,1,2,149,310,1,0,148,1,1.30,1,0,2,1
2,61,1,2,91,266,1,1,154,0,1.30,2,2,3,0
3,50,1,3,105,284,0,1,147,1,2.00,0,0,2,0
4,50,0,0,114,200,1,0,163,1,0.90,2,2,2,1
5,71,0,3,155,271,1,0,135,0,1.60,0,0,2,1
6,33,1,3,135,281,0,2,174,0,2.00,2,0,2,0
7,63,0,2,143,216,0,2,139,1,0.20,1,0,3,1
8,38,0,2,111,273,0,0,137,0,4.40,0,1,2,1
9,33,0,2,111,273,0,1,152,0,2.70,0,1,2,0


Button(button_style='info', description='Run Demo Prediction', icon='check', style=ButtonStyle())

Output()

Light theme dashboard is ready.
Dataset labels are synthetic practice labels.
